# Практическая работа №4: Временные ряды и пользовательские сессии
**Цель работы:** Научиться работать с датами и временем в pandas, выполнять агрегирование по временным интервалам и использовать оконные функции для анализа активности пользователей и их сессий на основе логов заказов и событий.

In [1]:
import pandas as pd

data_dir = 'data'

# Загружаем логи событий из events_lines.json
try:
    df_events = pd.read_json(f'{data_dir}/events_lines.json', lines=True)
    
    # Преобразуем текстовые даты в объекты datetime pandas
    df_events['event_time'] = pd.to_datetime(df_events['event_time'])
    
    # Извлекаем день недели
    df_events['day_of_week'] = df_events['event_time'].dt.day_name()
    
    print("Данные успешно загружены! Первые 3 строки:")
    print(df_events.head(3).to_string())
except FileNotFoundError:
    print("Файл events_lines.json не найден в папке data/")

Данные успешно загружены! Первые 3 строки:
   event_id  user_id          event_time   event_type         page day_of_week
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20     Tuesday
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart      Monday
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30     Tuesday


In [2]:
# Задание 1. Построить по событиям перекрестную таблицу (pivot)
pivot_events = pd.crosstab(index=df_events['day_of_week'], columns=df_events['event_type'])

# Сортируем дни недели по порядку
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot_events = pivot_events.reindex(days_order)

print("Количество событий по дням недели и типам:")
print(pivot_events.to_string())

Количество событий по дням недели и типам:
event_type   add_to_cart  click  page_view  purchase  scroll
day_of_week                                                 
Monday                53     73         41        55      61
Tuesday               47     55         48        49      49
Wednesday             60     57         58        68      46
Thursday              60     64         61        62      66
Friday                56     59         71        57      63
Saturday              54     50         48        64      60
Sunday                57     58         52        58      60


In [3]:
# Задание 2. Для каждого типа событий посчитать долю от общего количества событий в день.
daily_totals = pivot_events.sum(axis=1)
pivot_proportions = pivot_events.div(daily_totals, axis=0)

print("Доля каждого события от общего количества в день:")
print(pivot_proportions.round(4).to_string())

print("\nВ процентах (%):")
print((pivot_proportions * 100).round(2).to_string())

Доля каждого события от общего количества в день:
event_type   add_to_cart   click  page_view  purchase  scroll
day_of_week                                                  
Monday            0.1873  0.2580     0.1449    0.1943  0.2155
Tuesday           0.1895  0.2218     0.1935    0.1976  0.1976
Wednesday         0.2076  0.1972     0.2007    0.2353  0.1592
Thursday          0.1917  0.2045     0.1949    0.1981  0.2109
Friday            0.1830  0.1928     0.2320    0.1863  0.2059
Saturday          0.1957  0.1812     0.1739    0.2319  0.2174
Sunday            0.2000  0.2035     0.1825    0.2035  0.2105

В процентах (%):
event_type   add_to_cart  click  page_view  purchase  scroll
day_of_week                                                 
Monday             18.73  25.80      14.49     19.43   21.55
Tuesday            18.95  22.18      19.35     19.76   19.76
Wednesday          20.76  19.72      20.07     23.53   15.92
Thursday           19.17  20.45      19.49     19.81   21.09
Friday  

In [4]:
# Задание 3. Найти день недели, где доля add_to_cart максимальна
if 'add_to_cart' in pivot_proportions.columns:
    max_day = pivot_proportions['add_to_cart'].idxmax()
    max_value = pivot_proportions.loc[max_day, 'add_to_cart']
    
    print(f"⭐ День с максимальной долей 'add_to_cart': {max_day}")
    print(f"📊 Доля составляет: {max_value * 100:.2f}%")
else:
    print("Событие 'add_to_cart' не найдено.")

⭐ День с максимальной долей 'add_to_cart': Wednesday
📊 Доля составляет: 20.76%
